# Feature Engineering

This section creates analytical features from the cleaned dataset to support
sales, profitability, customer, time, and supply chain performance analysis.

## 1. Load Clean Dataset

In [33]:
import pandas as pd
import numpy as np

file_path = r"D:\DataCo_Smart_Supply_Chain_Analytics\data\processed\DataCo_cleaned.csv"

df = pd.read_csv(file_path)

df["order date (DateOrders)"] = pd.to_datetime(
    df["order date (DateOrders)"]
)

df["shipping date (DateOrders)"] = pd.to_datetime(
    df["shipping date (DateOrders)"]
)

## 2. Shipping Performance Features

Shipping performance metrics are created to evaluate delivery efficiency
and identify delayed shipments.

In [34]:
df["Shipping Delay"] = (
    df["Days for shipping (real)"]
    - df["Days for shipment (scheduled)"]
)

In [35]:
df["On Time Delivery"] = np.where(
    df["Shipping Delay"] <= 0,
    1,
    0
)

In [36]:
df["Late Delivery Flag"] = np.where(
    df["Shipping Delay"] > 0,
    1,
    0
)

## 3. Date Features

Calendar attributes are extracted from the order date to support
time-series and seasonal analysis.

In [38]:
df["Order Year"] = df["order date (DateOrders)"].dt.year

df["Order Quarter"] = (
    df["order date (DateOrders)"]
    .dt.quarter
)

df["Order Month"] = (
    df["order date (DateOrders)"]
    .dt.month
)

df["Order Month Name"] = (
    df["order date (DateOrders)"]
    .dt.month_name()
)

df["Order Year Month"] = (
    df["order date (DateOrders)"]
    .dt.to_period("M")
    .astype(str)
)

## 4. Profitability Features

Profitability metrics are derived to compare revenue generation with
profit performance across products, categories, markets, and regions.

In [39]:
df["Profit Margin"] = np.where(
    df["Sales"] != 0,
    df["Benefit per order"] / df["Sales"],
    0
)

## 5. Customer Features

Customer-level indicators are created to support customer segmentation
and repeat-purchase analysis.

In [40]:
customer_orders = (
    df.groupby("Customer Id")["Order Id"]
      .nunique()
)

df["Customer Order Count"] = (
    df["Customer Id"]
    .map(customer_orders)
)

In [41]:
df["Repeat Customer"] = np.where(
    df["Customer Order Count"] > 1,
    1,
    0
)

## 6. Feature Validation

The newly created features are reviewed to ensure that their values are
valid and suitable for downstream analysis.

In [42]:
feature_columns = [
    "Shipping Delay",
    "On Time Delivery",
    "Late Delivery Flag",
    "Order Year",
    "Order Quarter",
    "Order Month",
    "Profit Margin",
    "Customer Order Count",
    "Repeat Customer"
]

df[feature_columns].describe()

,Shipping Delay,On Time Delivery,Late Delivery Flag,Order Year,Order Quarter,Order Month,Profit Margin,Customer Order Count,Repeat Customer
count,180519.000000,180519.000000,180519.000000,180519.000000,180519.000000,180519.000000,180519.000000,180519.000000,180519.000000
mean,0.565807,0.427207,0.572793,2015.971150,2.417319,6.235449,0.108326,5.412943,0.944831
std,1.490966,0.494674,0.494674,0.829429,1.096876,3.403571,0.420594,2.339720,0.228310
min,-2.000000,0.000000,0.000000,2015.000000,1.000000,1.000000,-2.750000,1.000000,0.000000
25%,0.000000,0.000000,0.000000,2015.000000,1.000000,3.000000,0.062240,4.000000,1.000000
50%,1.000000,0.000000,1.000000,2016.000000,2.000000,6.000000,0.242512,5.000000,1.000000
75%,1.000000,1.000000,1.000000,2017.000000,3.000000,9.000000,0.336014,7.000000,1.000000
max,4.000000,1.000000,1.000000,2018.000000,4.000000,12.000000,0.500443,15.000000,1.000000


## 7. Save Feature-Engineered Dataset

The feature-engineered dataset is saved as the main analytical dataset
for Python, SQL, and Power BI.

In [43]:
output_path = r"D:\DataCo_Smart_Supply_Chain_Analytics\data\processed\DataCo_analytics.csv"

df.to_csv(output_path, index=False)